# Single-Band Sweep + ORx AGC - ADRV9026

Declare a **sweep plan** (`SWEEP` dict with per-block `zip` or `grid` combine),
preview it with `summarize_sweep_plan`, then at **every point** auto-level the ORx
and capture. LO / TX power / signal path can each be swept independently.

## About the ORx "AGC"

There is no trustworthy hardware AGC for the ORx here, so we auto-level in software
on the **captured-IQ peak** with a **railed-sample clip veto** (`RxDecPowerGet`
compresses the range and `RxGainGet` returns 0, so the gain index is tracked in
software). At every sweep point `autolevel_orx` starts at the gain floor and trims
the ORx gain index (clamped to the valid **185..255** window) until the captured
peak lands in the asymmetric band `[target - tol_down, target + tol_up]`
(defaults -1.0 +0.3/-0.6 dBFS) with `railed == 0`.

If the TX is too strong to fit the target even at the gain floor (185) it stops as a
**fatal** condition (reduce TX power); if the signal is too weak to reach the band
even at max gain (255) it **accepts 255** as the best achievable (`at_max_gain`) --
watch the `leveled` column and the red X's on the plot.

## 1. Parameters (edit me)

In [ ]:
from adrvtrx import TxChannel, RxChannel

# --- Profile -----------------------------------------------------------------
PROFILES = {
    "98_linksharing":  "ADRV9025Init_StdUseCase98_LinkSharing.profile",   # Tx 491.52 MSPS, Np=12
    "102_linksharing": "ADRV9025Init_StdUseCase102_LinkSharing.profile",  # Np=16
    "14_linksharing":  "ADRV9025Init_StdUseCase14_LinkSharing.profile",
    "50_linksharing":  "ADRV9025Init_StdUseCase50_LinkSharing.profile",
}
PROFILE = "98_linksharing"          # <-- choose here

INPUT_DIR = "C:/Users/ohammi/OneDrive - aus.edu/DualBand/input_100"

# --- Band wiring (bench: TX2 -> ORx2) ----------------------------------------
BANDS = [
    {"name": "band1", "tx": TxChannel.TX2, "orx": RxChannel.ORX2,
     "signal": f"{INPUT_DIR}/Signal1.txt"},
]

# --- Sweep plan (each block: mode "zip" | "grid"; blocks multiply) -----------
# Omit a block to hold config defaults for that dimension.
SWEEP = {
    "power_db": {"mode": "grid", "shared": [20.0, 15.0, 10.0, 5.0, 0.0]},
    # "freq": {"mode": "grid", "lo1_hz": [1.0e9, 1.1e9]},
    # "signals": {"mode": "grid", "band1": [f"{INPUT_DIR}/Signal1.txt", f"{INPUT_DIR}/Signal2.txt"]},
}

# --- ORx auto-level (AGC) ----------------------------------------------------
USE_AGC          = True
ORX_TARGET_DBFS  = -1.0
ORX_TOL_UP_DB    = 0.3
ORX_TOL_DOWN_DB  = 0.6
ORX_GAIN_INDEX   = 240     # used only when USE_AGC=False
CAPTURE_MS       = 0.05
OUTPUT_OVERSAMPLE = 2
SAVE_DIR          = None   # e.g. "captures/sweep"


## 2. Imports, config, profile

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from adrvtrx.config import load_config
from adrvtrx.profile import read_profile
from adrvtrx.radio import Radio
from adrvtrx.experiment import verify_status
from adrvtrx.capture import capture
from adrvtrx.gain import clip_report, autolevel_orx, ORX_GAIN_MIN, ORX_GAIN_MAX
from adrvtrx.sweep_plan import (
    flatten_point, format_point_label, run_planned_sweep,
    summarize_sweep_plan, sweep_defaults_from_config,
)
from adrvtrx.align import estimate_delay, estimate_and_align, match_corr

cfg = load_config()
cfg.profile_name = PROFILES.get(PROFILE, PROFILE)
SWEEP_DEFAULTS = sweep_defaults_from_config(cfg, BANDS)
info = read_profile(cfg.profile_path)
print(f"LO1 = {cfg.lo.lo1_hz/1e6:.1f} MHz | LO2 = {cfg.lo.lo2_hz/1e6:.1f} MHz")
print("Profile:", cfg.profile_path.name)
print(f"ORx auto-level (AGC) on captured-IQ peak; gain window {ORX_GAIN_MIN}..{ORX_GAIN_MAX}")
print(summarize_sweep_plan(BANDS, SWEEP, SWEEP_DEFAULTS))

## 3. Signal-path summary - sample rates & bit depth

In [ ]:
def _fs(bits):
    return (1 << (bits - 1)) - 1

print(f"{'path':<6}{'rate (MSPS)':>14}{'bits (Np)':>12}{'full scale':>12}")
print(f"{'TX':<6}{info.tx_rate_khz/1000:>14.3f}{info.tx_bits:>12}{_fs(info.tx_bits):>12}")
print(f"{'Rx':<6}{info.rx_rate_khz/1000:>14.3f}{info.rx_bits:>12}{_fs(info.rx_bits):>12}")
print(f"{'ORx':<6}{info.orx_rate_khz/1000:>14.3f}{info.rx_bits:>12}{_fs(info.rx_bits):>12}")

## 4. Signals

Waveforms load **per sweep point** inside `run_planned_sweep` (cached by file path).
All active TX buffers must be the same length.

In [ ]:
# (no preload — see run_planned_sweep in section 7)

## 5. Connect, program, verify

In [ ]:
radio = Radio(cfg)
radio.connect()
radio.force_safe()
radio.program()
for k, v in verify_status(radio).items():
    print(f"{k}: {v}")

## 6. Build the sweep axis + per-point action (auto-level ORx, capture)

In [ ]:
band = BANDS[0]
ORX_CHANNEL = band["orx"]
TX_CHANNEL = band["tx"]

def measure(ch):
    cap = capture(radio, int(ch), CAPTURE_MS, bits=info.rx_bits).channels[ch]
    rep = clip_report(cap.i, cap.q, info.rx_bits)
    return rep.peak_dbfs, rep.railed_samples

def action(point, waves):
    flat = flatten_point(point)
    label = format_point_label(point)
    ref = waves[TX_CHANNEL]
    if USE_AGC:
        lr = autolevel_orx(
            lambda g: radio.set_rx_gain(ORX_CHANNEL, g),
            lambda: measure(ORX_CHANNEL),
            target_dbfs=ORX_TARGET_DBFS, tol_up_db=ORX_TOL_UP_DB, tol_down_db=ORX_TOL_DOWN_DB,
        )
        orx_gain, leveled, note = lr.final_gain_index, lr.converged, lr.reason
    else:
        radio.set_rx_gain(ORX_CHANNEL, ORX_GAIN_INDEX)
        orx_gain, leveled, note = ORX_GAIN_INDEX, True, "manual"
    out_ms = OUTPUT_OVERSAMPLE * len(ref) / info.orx_rate_hz * 1e3
    out = capture(radio, int(ORX_CHANNEL), out_ms, bits=info.rx_bits).channels[ORX_CHANNEL]
    rep = clip_report(out.i, out.q, info.rx_bits)
    saved = ""
    if SAVE_DIR:
        from pathlib import Path
        Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
        saved = f"{SAVE_DIR}/{label}.txt"
        out.save(saved)
    y = out.i.astype(float) + 1j * out.q.astype(float)
    delay = estimate_delay(ref, y, info.orx_rate_hz)
    corr = match_corr(ref, y, info.orx_rate_hz)
    rec = {**flat, orx_gain=orx_gain, leveled=leveled, peak_dBFS=rep.peak_dbfs,
           railed=rep.railed_samples, delay_samples=delay,
           delay_ns=delay / info.orx_rate_hz * 1e9, corr=corr, saved=saved, note=note}
    print(f"  {label}: gain {orx_gain}, peak {rep.peak_dbfs:+.1f} dBFS, "
          f"delay {delay:.2f} samp ({rec['delay_ns']:.1f} ns), corr {corr:.3f}"
          + (f" -> {saved}" if saved else ""))
    return rec

## 7. Run the sweep

In [ ]:
records = run_planned_sweep(
    radio, BANDS, SWEEP, action, defaults=SWEEP_DEFAULTS, tx_bits=info.tx_bits,
)
radio.disable_tx()

def _xkey(records):
    for k in records[0]:
        if k.endswith("_power_db") or k.endswith("_hz"):
            if len({r[k] for r in records}) > 1:
                return k
    return None

XKEY = _xkey(records) or "point"
XLABEL = XKEY.replace("_", " ")
hdr = f"\n{'#':>4} {XLABEL:>16} {'orx_gain':>9} {'peak_dBFS':>10} {'delay_ns':>9} {'corr':>6} {'leveled':>8}"
print(hdr)
for i, r in enumerate(records):
    x = r.get(XKEY, i)
    print(f"{i:>4} {str(x):>16} {r['orx_gain']:>9} {r['peak_dBFS']:>10.1f} "
          f"{r['delay_ns']:>9.1f} {r['corr']:>6.3f} {str(r['leveled']):>8}")

## 8. Plot - AGC holds the level while gain compensates

In [ ]:
if XKEY in records[0]:
    x = np.array([r[XKEY] for r in records], float)
else:
    x = np.arange(len(records))
gain = np.array([r["orx_gain"] for r in records])
peak = np.array([r["peak_dBFS"] for r in records])
ok = np.array([r["leveled"] for r in records], bool)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(x, gain, "o-")
ax[0].set_title("settled ORx gain index"); ax[0].set_xlabel(XLABEL)
ax[0].set_ylabel("gain index"); ax[0].grid(True)
ax[1].axhline(ORX_TARGET_DBFS, ls="--", c="k", lw=1, label="target")
ax[1].plot(x[ok], peak[ok], "o-", c="g", label="leveled")
if (~ok).any():
    ax[1].plot(x[~ok], peak[~ok], "x", c="r", ms=9, label="NOT leveled")
ax[1].set_title("captured peak after AGC"); ax[1].set_xlabel(XLABEL)
ax[1].set_ylabel("dBFS"); ax[1].legend(); ax[1].grid(True)
plt.tight_layout(); plt.show()

## 9. Safe-state & disconnect

In [ ]:
radio.safe_state()
radio.disconnect()
print("TX safe, board disconnected")